# Question 8 - 122. Best Time to Buy and Sell Stock II

You are given an integer array `prices` where `prices[i]` is the price of a given stock on the `i-th` day.

On each day, you may decide to buy and/or sell the stock. You can only hold at most one share of the stock at any time. However, you can buy it then immediately sell it on the same day.

Find and return the **maximum profit** you can achieve.

**Example 1:**

    Input: prices = [7,1,5,3,6,4]
    Output: 7
    Explanation: Buy on day 2 (price = 1) and sell on day 3 (price = 5), profit = 5-1 = 4.
    Then buy on day 4 (price = 3) and sell on day 5 (price = 6), profit = 6-3 = 3.
    Total profit is 4 + 3 = 7.

**Example 2:**

    Input: prices = [1,2,3,4,5]
    Output: 4
    Explanation: Buy on day 1 (price = 1) and sell on day 5 (price = 5), profit = 5-1 = 4.
    Total profit is 4.

**Example 3:**

    Input: prices = [7,6,4,3,1]
    Output: 0
    Explanation: There is no way to make a positive profit, so we never buy the stock to achieve the maximum profit of 0.

**Constraints:**

- `1 <= prices.length <= 3 * 10^4`
- `0 <= prices[i] <= 10^4`


---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Greedy (sum of positive daily gains)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force | O(2ⁿ) | O(n) |
| 2️⃣ Optimized | O(n) | O(1) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding


🗣️ *"Clarifying:"*

- **Unlimited** transactions, but I can hold at most one share at a time? → **Yes.**
- Can I sell and re-buy on the same day? → **Yes.** That's why the greedy works.
- Return the max total profit. 0 if nothing helps.

### Step 1 · Brute force


**Idea:** Try every decision sequence. On each day, if I'm not holding, I can **buy** or **skip**. If I'm holding, I can **sell** or **skip**. Recurse over `(day, holding)`.

🗣️ *"The brute force explores every buy/sell/skip choice recursively, which is exponential: O(2ⁿ). Memoizing on `(day, holding)` would make it O(n) DP. But first let me look for structure that makes it even simpler."*

In [1]:
class SolutionBrute:
    def maxProfit(self, prices: list[int]) -> int:
        def dfs(day: int, holding: bool) -> int:
            if day == len(prices):
                return 0
            skip = dfs(day + 1, holding)
            if holding:
                act = prices[day] + dfs(day + 1, False)   # sell
            else:
                act = -prices[day] + dfs(day + 1, True)   # buy
            return max(skip, act)
        return dfs(0, False)

### Step 2 · Optimize


**Bottleneck:** Recursion re-explores the same `(day, holding)` states again and again.

**Key insight:** Any profitable hold from day `i` to day `j` equals the sum of the daily changes in between:
`p[j] − p[i] = (p[i+1]−p[i]) + … + (p[j]−p[j−1])`.
With unlimited trades, we should collect **every positive daily change** and skip every negative one. That's the same as buying before each up-day and selling after it.

🗣️ *"Since I can trade as often as I like, any long rise splits into day-by-day gains. So the max profit is the sum of all positive differences `prices[i] − prices[i−1]`. I never want a down-day inside a hold, because I could sell before it and re-buy after. That's one pass, O(n) time and O(1) space."*

In [2]:
class Solution:
    def maxProfit(self, prices: list[int]) -> int:
        profit = 0
        for i in range(1, len(prices)):
            if prices[i] > prices[i - 1]:
                profit += prices[i] - prices[i - 1]
        return profit

### Step 3 · Toy example walkthrough


`prices = [7, 1, 5, 3, 6, 4]`

| i | change prices[i] − prices[i−1] | take it? | profit |
|---|---|---|---|
| 1 | 1 − 7 = −6 | ❌ | 0 |
| 2 | 5 − 1 = **+4** | ✅ | 4 |
| 3 | 3 − 5 = −2 | ❌ | 4 |
| 4 | 6 − 3 = **+3** | ✅ | 7 |
| 5 | 4 − 6 = −2 | ❌ | 7 |

Answer `7` = (buy 1 → sell 5) + (buy 3 → sell 6) ✅

For `[1, 2, 3, 4, 5]`: +1 +1 +1 +1 = 4, the same as one trade buying at 1 and selling at 5.

In [3]:
cases = [
    ([7, 1, 5, 3, 6, 4], 7),
    ([1, 2, 3, 4, 5], 4),
    ([7, 6, 4, 3, 1], 0),
    ([5], 0),
    ([2, 1, 2, 0, 1], 2),
]
for prices, expected in cases:
    assert SolutionBrute().maxProfit(prices) == expected
    assert Solution().maxProfit(prices) == expected
    print(prices, "->", expected)
print("All tests passed ✅")

[7, 1, 5, 3, 6, 4] -> 7
[1, 2, 3, 4, 5] -> 4
[7, 6, 4, 3, 1] -> 0
[5] -> 0
[2, 1, 2, 0, 1] -> 2
All tests passed ✅


### Step 4 · Wrap up


| | Time | Space |
|---|---|---|
| Brute force (recursion) | O(2ⁿ) | O(n) stack |
| DP on (day, holding) | O(n) | O(1) with two variables |
| Optimized (greedy) | O(n) | O(1) |

**DP version (good to mention, since it generalises to fees and cooldown):**
`cash = max(cash, hold + p)`, `hold = max(hold, cash - p)`.

**Follow-ups:** a transaction fee (LC 714) breaks the greedy. Use the cash/hold DP and subtract the fee on sell.

🗣️ **30-second recap:** *"Brute force tries every buy/sell/skip sequence, which is exponential. Because trades are unlimited, any rising stretch can be split into daily gains, so the answer is the sum of all positive day-to-day differences. That's O(n) time and O(1) space. If a fee or cooldown is added, I'd switch to the two-state DP."*

---
## 📎 Appendix: original reference solution

*Kept from the source repo for comparison.*

In [3]:
class Solution:
    # solve with O(n) time complexity and O(1) space complexity
    def maxProfit(self, prices: list[int]) -> int:
        # init profit to 0
        profit = 0

        # iterate through the list of prices
        for i in range(1, len(prices)):
            # if the price on the current day is higher than the previous day
            if prices[i] > prices[i - 1]:
                # add the difference to the profit
                profit += prices[i] - prices[i - 1]

        # return the profit
        return profit

In [4]:
maxProfit = Solution().maxProfit

print(maxProfit([7, 1, 5, 3, 6, 4]))  # 7
print(maxProfit([1, 2, 3, 4, 5]))  # 4
print(maxProfit([7, 6, 4, 3, 1]))  # 0

7
4
0
